# Análisis exploratorio y modelaje de datos
**Optimización del portafolio de variedades y la asignación de superficie para maximizar el margen por metro cuadrado: el caso de una florícola exportadora de rosas**

Bryan Steve Vargas Cabascango — Maestría en Inteligencia de Negocios y Ciencia de Datos

---

Este cuaderno reproduce íntegramente el análisis exploratorio del proyecto. Se ejecuta de arriba hacia abajo y genera cada figura de forma individual.

**Requisitos:** `pandas`, `numpy`, `matplotlib`, `openpyxl`, `scikit-learn`.

```
pip install pandas numpy matplotlib openpyxl scikit-learn
```

## 1. Configuración

In [ ]:
import warnings, os, json
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- RUTAS: ajustar a la carpeta donde estan los tres archivos -----------
RUTA = './datos/'
SALIDA = './figuras/'
os.makedirs(SALIDA, exist_ok=True)

ARCH_PANEL  = RUTA + 'dbo_panel_tesis_db_v1_20260913.xlsx'   # panel mensual
ARCH_COSTOS = RUTA + 'dbo_costs_v1.xlsx'
ARCH_PLAN   = RUTA + 'dbo_crop_plan_clean_mo.xlsx'

pd.set_option('display.width', 180, 'display.max_columns', 60)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10,
                     'axes.grid': True, 'grid.alpha': .3, 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False})

AZ, RO, VE, MO, NA = '#4C72B0', '#C44E52', '#55A868', '#8172B2', '#DD8452'

def guardar(fig, nombre):
    """Guarda cada figura de forma individual."""
    fig.tight_layout()
    fig.savefig(f'{SALIDA}{nombre}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. Identificación y carga de las bases de datos

Se utilizan tres fuentes primarias internas de la organización, todas con periodicidad mensual y cobertura de enero de 2020 a agosto de 2026.

In [ ]:
panel = pd.read_excel(ARCH_PANEL)
costos = pd.read_excel(ARCH_COSTOS)
plan = pd.read_excel(ARCH_PLAN).rename(columns={'clean_variety.1': 'variety'})

for df_ in (panel, costos, plan):
    df_['id_month'] = pd.to_datetime(df_['id_month'])

print('PANEL DE PRODUCCION Y VENTAS')
print(f'  {len(panel):,} filas | {panel.variety.nunique()} variedades | '
      f'{panel.id_month.nunique()} meses')
print(f'  columnas: {list(panel.columns)}\n')
print('ESTADO DE COSTOS (nivel finca)')
print(f'  {len(costos):,} filas | columnas: {list(costos.columns)}\n')
print('PLAN DE CULTIVO')
print(f'  {len(plan):,} filas | {plan.variety.nunique()} variedades')
print(f'  columnas: {list(plan.columns)}')

In [ ]:
# Valores faltantes y rango temporal de cada fuente
for nombre, df_ in [('panel', panel), ('costos', costos), ('plan', plan)]:
    nulos = df_.isna().sum()
    print(f'{nombre}: {df_.id_month.min():%Y-%m} a {df_.id_month.max():%Y-%m} | '
          f'nulos: {dict(nulos[nulos > 0]) if nulos.sum() else "ninguno"}')

## 3. Correcciones y transformaciones

Se aplican cinco procedimientos de depuración. Cada uno se documenta con el número de registros afectados.

In [ ]:
registro = {}

# 3.1 Exclusion de productos que no son variedades de rosa
NO_ROSA = ['Mix Alstroemeria', 'Codigo', 'Mix Spray Roses', 'Mixto', 'Bouquet']
registro['filas_iniciales'] = len(panel)
panel = panel[~panel.variety.isin(NO_ROSA)].copy()
plan = plan[~plan.variety.isin(NO_ROSA)].copy()
registro['excluidas_no_rosa'] = registro['filas_iniciales'] - len(panel)

# 3.2 Duplicados en el plan de cultivo
#     127 pares variedad-mes repetidos por un error de etiquetado del campo
#     'year' en enero de 2025; los valores son identicos, se conserva el primero
dup = plan.duplicated(['variety', 'id_month']).sum()
plan = plan.drop_duplicates(subset=['variety', 'id_month'], keep='first')
registro['duplicados_plan'] = int(dup)

print(json.dumps(registro, indent=1))

In [ ]:
# 3.3 Integracion de las fuentes por la clave variedad-mes
df = panel.merge(
    plan[['variety', 'id_month', 'n_plants', 'area', 'density',
          'monthly_flowers_per_plant']],
    on=['variety', 'id_month'], how='left')

registro['sin_plan_cultivo'] = int(df.area.isna().sum())
df = df[(df.area > 0) & (df.n_plants > 0)].copy()
print(f"Registros sin plan de cultivo descartados: {registro['sin_plan_cultivo']}")
print(f'Filas tras la integración: {len(df):,}')

In [ ]:
# 3.4 Control de calidad: produccion incompatible con la capacidad fisica
#     ratio = productividad observada / capacidad planificada (densidad x flores por planta)
df['ratio_prod'] = (df.gross_prod / df.area) / (df.monthly_flowers_per_plant * df.density)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df.ratio_prod.clip(0, 6), bins=60, color=AZ, edgecolor='white', lw=.4)
ax.axvline(3, color=RO, ls='--', lw=1.6, label='Umbral de descarte (3x)')
ax.set_xlabel('Producción observada / capacidad planificada')
ax.set_ylabel('Observaciones')
ax.set_title('Control de calidad: consistencia física de la producción')
ax.legend()
guardar(fig, 'fig00_control_calidad')

registro['descartadas_calidad'] = int((df.ratio_prod > 3).sum())
df = df[df.ratio_prod <= 3].copy()
print(f"Descartadas por inconsistencia física: {registro['descartadas_calidad']} "
      f"({registro['descartadas_calidad'] / len(df) * 100:.1f}%)")

### 3.5 Imputación de costos por variedad

La contabilidad registra los costos de forma agregada. Se identifican dos inductores que varían entre variedades:

| Componente | Inductor | Fundamento |
|---|---|---|
| Costo de campo | Plantas-mes | El mantenimiento escala con el número de plantas, que difiere por la densidad de siembra |
| Costo de poscosecha | Tallos brutos procesados | La clasificación y el empaque escalan con el volumen físico que ingresa al proceso |

$$C_{v,t} = \bar{C}^f_t \cdot \frac{n_{v,t}}{\sum_j n_{j,t}} + \bar{C}^p_t \cdot \frac{q_{v,t}}{\sum_j q_{j,t}}
\qquad\qquad
m_{v,t} = \frac{S_{v,t} - C_{v,t}}{a_{v,t}}$$

In [ ]:
tot = df.groupby('id_month').agg(plants_t=('n_plants', 'sum'),
                                 gross_t=('gross_prod', 'sum')).reset_index()
df = df.merge(tot, on='id_month').merge(
    costos[['id_month', 'farm_cost', 'pharvest_cost', 'total_cost']], on='id_month')

df['cost_farm_v'] = df.farm_cost * df.n_plants / df.plants_t
df['cost_ph_v']   = df.pharvest_cost * df.gross_prod / df.gross_t
df['cost_v']      = df.cost_farm_v + df.cost_ph_v

df['margin']          = df.sales_usd - df.cost_v
df['margin_m2']       = df.margin / df.area
df['rev_m2']          = df.sales_usd / df.area
df['cost_m2']         = df.cost_v / df.area
df['stems_m2']        = df.gross_prod / df.area
df['pct_nacional']    = df.national / df.gross_prod
df['pct_no_colocada'] = 1 - df.sales / df.gross_prod.replace(0, np.nan)

# Verificacion de reconciliacion con la contabilidad
rec = df.groupby('id_month').cost_v.sum().to_frame('imputado').join(
    costos.set_index('id_month').total_cost)
dif = ((rec.imputado / rec.total_cost - 1) * 100).abs().max()
print(f'Diferencia máxima de reconciliación: {dif:.2e} %')
assert dif < 1e-6, 'La imputación no reconcilia con la contabilidad'

In [ ]:
# 3.6 Winsorizacion de la variable de resultado en los percentiles 1 y 99
lo, hi = df.margin_m2.quantile([.01, .99])
registro['winsorizadas'] = int(((df.margin_m2 < lo) | (df.margin_m2 > hi)).sum())
df['margin_m2'] = df.margin_m2.clip(lo, hi)
print(f'Winsorización en [{lo:.3f}, {hi:.3f}] — {registro["winsorizadas"]} valores acotados')

### 3.7 Definición del universo de variedades

Siguiendo el criterio de Barkley et al. (2010), se conservan únicamente las variedades con historial productivo suficiente.

In [ ]:
cob = df.groupby('variety').agg(meses=('id_month', 'nunique'),
                                area_med=('area', 'mean'),
                                ventas=('sales_usd', 'sum'),
                                ultimo=('id_month', 'max')).reset_index()

MIN_MESES, MIN_AREA = 48, 200.0
cob['incluida'] = ((cob.meses >= MIN_MESES) & (cob.area_med >= MIN_AREA)
                   & (cob.ultimo >= '2026-01-01'))

sel = cob.loc[cob.incluida, 'variety'].tolist()
d = df[df.variety.isin(sel)].copy()
d = d.rename(columns={'is_valentines_season`': 'is_valentin'})

print(f'Variedades totales:   {len(cob)}')
print(f'Variedades incluidas: {len(sel)}')
print(f'Cobertura de ventas:      {cob[cob.incluida].ventas.sum() / cob.ventas.sum() * 100:.1f} %')
print(f'Cobertura de superficie:  {cob[cob.incluida].area_med.sum() / cob.area_med.sum() * 100:.1f} %')
print(f'Observaciones del panel analítico: {len(d):,}')

## 4. Matriz de descripción de variables

In [ ]:
VARIABLES = [
 ('margin_m2','Construida','Continua','Variable dependiente. Margen bruto mensual por metro cuadrado, en dólares.'),
 ('rev_m2','Construida','Continua','Ingreso por metro cuadrado al mes.'),
 ('cost_m2','Construida','Continua','Costo imputado por metro cuadrado al mes.'),
 ('stems_m2','Construida','Continua','Tallos brutos producidos por metro cuadrado al mes.'),
 ('pct_nacional','Construida','Continua','Fracción de la producción clasificada como nacional y descartada.'),
 ('pct_no_colocada','Construida','Continua','Fracción de la producción bruta que no encuentra comprador.'),
 ('avg_price','Panel de ventas','Continua','Precio promedio por tallo vendido, en dólares.'),
 ('gross_prod','Panel de ventas','Continua','Producción bruta de tallos en el mes.'),
 ('exportable','Panel de ventas','Continua','Tallos de calidad exportable.'),
 ('national','Panel de ventas','Continua','Tallos de calidad nacional, descartados íntegramente.'),
 ('sales','Panel de ventas','Continua','Tallos efectivamente vendidos.'),
 ('discarted','Panel de ventas','Continua','Tallos dados de baja.'),
 ('sales_usd','Panel de ventas','Continua','Ventas del mes en dólares.'),
 ('area','Plan de cultivo','Continua','Superficie productiva asignada a la variedad, en m².'),
 ('n_plants','Plan de cultivo','Discreta','Número de plantas sembradas de la variedad.'),
 ('density','Plan de cultivo','Continua','Plantas por metro cuadrado.'),
 ('monthly_flowers_per_plant','Plan de cultivo','Continua','Producción mensual esperada por planta.'),
 ('farm_cost','Estado de costos','Continua','Costo mensual de campo de la finca.'),
 ('pharvest_cost','Estado de costos','Continua','Costo mensual de poscosecha de la finca.'),
 ('season_id','Panel de ventas','Categórica','Temporada comercial: regular, San Valentín, Madres, Difuntos, Navidad.'),
 ('is_valentin','Panel de ventas','Binaria','Indicador de la temporada de San Valentín.'),
 ('is_mothers_day_season','Panel de ventas','Binaria','Indicador de la temporada del Día de la Madre.'),
 ('is_all_saints_season','Panel de ventas','Binaria','Indicador de la temporada de Difuntos.'),
 ('is_christmas_season','Panel de ventas','Binaria','Indicador de la temporada de Navidad.'),
 ('is_covid_shock','Panel de ventas','Binaria','Indicador de los meses de interrupción comercial de 2020.'),
 ('is_censored','Panel de ventas','Binaria','Indicador de que la variedad vendió toda su producción exportable.'),
]
matriz = pd.DataFrame(VARIABLES, columns=['Variable','Fuente','Tipo','Descripción'])
matriz.to_csv(SALIDA + 'matriz_variables.csv', index=False)
matriz

In [ ]:
# Estadistica descriptiva de las variables continuas
CONT = ['margin_m2','rev_m2','cost_m2','stems_m2','pct_nacional','pct_no_colocada',
        'avg_price','gross_prod','exportable','sales','sales_usd','area',
        'n_plants','density','monthly_flowers_per_plant']
desc = d[CONT].describe().T[['count','mean','std','min','50%','max']]
desc.columns = ['N','Media','Desv. est.','Mínimo','Mediana','Máximo']
desc.round(3)

## 5. Matriz de correlación de todas las variables

In [ ]:
ETIQUETAS = {'margin_m2':'Margen/m²','rev_m2':'Ingreso/m²','cost_m2':'Costo/m²',
  'stems_m2':'Tallos/m²','avg_price':'Precio medio','pct_nacional':'Prop. nacional',
  'pct_no_colocada':'Prop. no colocada','gross_prod':'Producción bruta',
  'exportable':'Exportable','sales':'Tallos vendidos','sales_usd':'Ventas USD',
  'area':'Superficie','n_plants':'N.º de plantas','density':'Densidad',
  'monthly_flowers_per_plant':'Flores/planta','is_valentin':'San Valentín',
  'is_mothers_day_season':'Día de la Madre','is_christmas_season':'Navidad',
  'is_covid_shock':'Choque COVID'}

X = d[list(ETIQUETAS)].rename(columns=ETIQUETAS)
C = X.corr()
C.round(3).to_csv(SALIDA + 'matriz_correlacion.csv')
C.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(C.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(C))); ax.set_yticks(range(len(C)))
ax.set_xticklabels(C.columns, rotation=90, fontsize=8)
ax.set_yticklabels(C.index, fontsize=8)
for i in range(len(C)):
    for j in range(len(C)):
        val = C.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=5.6,
                color='white' if abs(val) > .55 else 'black')
ax.grid(False)
plt.colorbar(im, ax=ax, shrink=.75)
ax.set_title('Matriz de correlación de las variables del estudio')
guardar(fig, 'fig12_correlacion_variables')

In [ ]:
# Correlaciones con la variable dependiente, ordenadas por magnitud
C['Margen/m²'].drop('Margen/m²').sort_values(key=abs, ascending=False).round(3)

## 6. Agregados por variedad

La unidad de decisión del proyecto es la variedad, por lo que se construye un resumen a ese nivel.

In [ ]:
v = d.groupby('variety').agg(
    meses=('id_month','nunique'), area=('area','mean'),
    margen=('margin_m2','mean'), sd=('margin_m2','std'),
    ingreso=('rev_m2','mean'), costo=('cost_m2','mean'),
    tallos=('stems_m2','mean'), precio=('avg_price','mean'),
    nacional=('pct_nacional','mean'), nocol=('pct_no_colocada','mean'),
    ventas=('sales_usd','sum'), margen_usd=('margin','sum')).reset_index()
v['anual'] = v.margen * 12
v['share'] = v.area / v.area.sum()
v['rentable'] = v.margen > 0
v.to_csv(SALIDA + 'descriptivo_variedades.csv', index=False)

print(f'Variedades con margen negativo: {(~v.rentable).sum()} de {len(v)}')
print(f'Superficie que ocupan: {v[~v.rentable].share.sum() * 100:.1f} %')
v.sort_values('anual', ascending=False).head(10).round(3)

## 7. Visualizaciones

Cada figura se genera de forma individual.

### 7.1 Evolución del margen agregado de la finca

In [ ]:
s = d.groupby('id_month').apply(lambda g: g.margin.sum() / g.area.sum())

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(s.index, s.values, color=AZ, lw=1.6)
ax.axhline(0, color='k', lw=.8)
ax.fill_between(s.index, 0, s.values, where=s.values < 0, color=RO, alpha=.25)
ax.set_ylabel('USD por m² al mes')
ax.set_title('Margen mensual por metro cuadrado de la finca, 2020-2026')
guardar(fig, 'fig01_serie_margen')

### 7.2 Distribución del margen por metro cuadrado

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(d.margin_m2, bins=60, color=AZ, edgecolor='white', lw=.4)
ax.axvline(0, color='k', lw=1)
ax.axvline(d.margin_m2.mean(), color=RO, ls='--', lw=1.5, label=f'Media = {d.margin_m2.mean():.2f}')
ax.axvline(d.margin_m2.median(), color=VE, ls='--', lw=1.5, label=f'Mediana = {d.margin_m2.median():.2f}')
ax.set_xlabel('Margen (USD por m² al mes)'); ax.set_ylabel('Observaciones')
ax.set_title('Distribución del margen por metro cuadrado')
ax.legend()
guardar(fig, 'fig02_distribucion_margen')

### 7.3 Variedades con mayor margen

In [ ]:
t = v.nlargest(15, 'anual').sort_values('anual')
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(t.variety, t.anual, color=VE)
ax.set_xlabel('Margen (USD por m² al año)')
ax.set_title('Quince variedades con mayor margen por metro cuadrado')
guardar(fig, 'fig03_top15_margen')

### 7.4 Variedades con menor margen

In [ ]:
t = v.nsmallest(15, 'anual').sort_values('anual')
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(t.variety, t.anual, color=RO)
ax.set_xlabel('Margen (USD por m² al año)')
ax.set_title('Quince variedades con menor margen por metro cuadrado')
guardar(fig, 'fig04_bottom15_margen')

### 7.5 Riesgo y retorno por variedad

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(v.sd * np.sqrt(12), v.anual, s=v.area / 45, alpha=.6,
           c=np.where(v.rentable, VE, RO), edgecolor='k', lw=.4)
ax.axhline(0, color='k', lw=.9)
ax.set_xlabel('Riesgo: desviación estándar anualizada (USD por m²)')
ax.set_ylabel('Margen esperado (USD por m² al año)')
ax.set_title('Riesgo y retorno por variedad (el tamaño indica la superficie)')
guardar(fig, 'fig05_riesgo_retorno')

### 7.6 Superficie según el signo del margen

In [ ]:
g = v.groupby('rentable').agg(sup=('area','sum'), n=('variety','count'))
fig, ax = plt.subplots(figsize=(6, 4.2))
b = ax.bar(['Margen negativo','Margen positivo'],
           [g.loc[False,'sup'], g.loc[True,'sup']], color=[RO, VE], width=.55)
for r, n, sp in zip(b, [g.loc[False,'n'], g.loc[True,'n']],
                    [g.loc[False,'sup'], g.loc[True,'sup']]):
    ax.text(r.get_x()+r.get_width()/2, sp, f'{n} variedades\n{sp/v.area.sum()*100:.1f}%',
            ha='center', va='bottom', fontsize=9.5)
ax.set_ylabel('Superficie media (m²)'); ax.set_ylim(0, v.area.sum()*.95)
ax.set_title('Superficie asignada según el signo del margen')
guardar(fig, 'fig06_superficie_por_signo')

### 7.7 Margen por temporada comercial

In [ ]:
orden = ['REGULAR','VALENTIN','MADRES','DIFUNTOS','NAVIDAD']
e = d.groupby('season_id').margin_m2.mean().reindex(orden) * 12
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(orden, e.values, color=[MO if x > 0 else RO for x in e.values], width=.6)
ax.axhline(0, color='k', lw=.9)
ax.set_ylabel('Margen (USD por m² al año)')
ax.set_title('Margen medio según la temporada comercial')
ax.tick_params(axis='x', rotation=15)
guardar(fig, 'fig07_temporada')

### 7.8 Perfil estacional por mes calendario

In [ ]:
m = d.groupby(d.id_month.dt.month).margin_m2.mean() * 12
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 13), m.values, color=AZ, width=.65)
ax.axhline(0, color='k', lw=.9)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'])
ax.set_ylabel('Margen (USD por m² al año)')
ax.set_title('Perfil estacional del margen por mes calendario')
guardar(fig, 'fig08_estacionalidad_mes')

### 7.9 Flor no colocada frente al margen

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(v.nocol*100, v.anual, s=v.area/45, alpha=.6,
           c=np.where(v.rentable, VE, RO), edgecolor='k', lw=.4)
z = np.polyfit(v.nocol*100, v.anual, 1)
xs = np.linspace((v.nocol*100).min(), (v.nocol*100).max(), 50)
ax.plot(xs, np.polyval(z, xs), 'k--', lw=1.2, label=f'Tendencia (r = {v.nocol.corr(v.anual):.2f})')
ax.axhline(0, color='k', lw=.9)
ax.set_xlabel('Producción no colocada (%)'); ax.set_ylabel('Margen (USD por m² al año)')
ax.set_title('Relación entre flor no colocada y margen')
ax.legend()
guardar(fig, 'fig09_nocolocada_vs_margen')

### 7.10 Ingreso frente a costo imputado

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(v.costo, v.ingreso, s=v.area/45, alpha=.6,
           c=np.where(v.rentable, VE, RO), edgecolor='k', lw=.4)
lim = [0, max(v.costo.max(), v.ingreso.max())*1.05]
ax.plot(lim, lim, 'k--', lw=1.2, label='Punto de equilibrio')
ax.set_xlabel('Costo imputado (USD por m² al mes)')
ax.set_ylabel('Ingreso (USD por m² al mes)')
ax.set_title('Ingreso frente a costo imputado por variedad')
ax.legend()
guardar(fig, 'fig10_ingreso_vs_costo')

### 7.11 Concentración de la superficie

In [ ]:
t = v.nlargest(15, 'area').sort_values('area')
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(t.variety, t.share*100, color=[VE if r else RO for r in t.rentable])
ax.set_xlabel('Participación en la superficie total (%)')
ax.set_title('Quince variedades con mayor superficie asignada')
guardar(fig, 'fig11_concentracion_superficie')

### 7.12 Correlación del margen entre variedades

In [ ]:
W = d[d.id_month >= '2022-01-01'].pivot_table(index='id_month', columns='variety', values='margin_m2')
W = W.loc[:, W.notna().mean() >= .95]
CV = W.corr()
off = CV.values[np.triu_indices_from(CV, 1)]

fig, ax = plt.subplots(figsize=(7.5, 6.2))
im = ax.imshow(CV.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.colorbar(im, ax=ax, shrink=.8)
ax.set_title(f'Correlación del margen entre las {CV.shape[0]} variedades\n(media = {np.nanmean(off):.3f})')
guardar(fig, 'fig13_correlacion_variedades')

### 7.13 Distribución de las correlaciones cruzadas

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(off, bins=45, color=AZ, edgecolor='white', lw=.4)
ax.axvline(0, color='k', lw=1)
ax.axvline(np.nanmean(off), color=RO, ls='--', lw=1.5, label=f'Media = {np.nanmean(off):.3f}')
ax.set_xlabel('Coeficiente de correlación entre pares de variedades')
ax.set_ylabel('Número de pares')
ax.set_title('Distribución de las correlaciones entre variedades')
ax.legend()
guardar(fig, 'fig14_histograma_correlaciones')
print(f'Pares con correlación negativa: {np.nanmean(off < 0)*100:.1f} %')

### 7.14 Descomposición de la varianza del margen

In [ ]:
gm = d.margin_m2.mean(); ss = ((d.margin_m2 - gm)**2).sum()
b_var = d.groupby('variety').margin_m2.agg(['mean','count']).eval('(mean-@gm)**2*count').sum()/ss
b_mes = d.groupby('id_month').margin_m2.agg(['mean','count']).eval('(mean-@gm)**2*count').sum()/ss

fig, ax = plt.subplots(figsize=(6.5, 4))
vals = [b_var*100, b_mes*100, (1-b_var-b_mes)*100]
ax.bar(['Entre variedades','Entre meses','Residual'], vals,
       color=[VE, MO, '#BBBBBB'], width=.55)
for i, val in enumerate(vals):
    ax.text(i, val, f'{val:.1f}%', ha='center', va='bottom')
ax.set_ylabel('Porcentaje de la varianza total')
ax.set_title('Descomposición de la varianza del margen por metro cuadrado')
guardar(fig, 'fig15_descomposicion_varianza')

### 7.15 Evolución de la superficie productiva

In [ ]:
a = df.groupby('id_month').area.sum()
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(a.index, a.values/1000, color=VE, lw=1.8)
ax.set_ylabel('Superficie total (miles de m²)')
ax.set_title('Evolución de la superficie productiva de la finca')
guardar(fig, 'fig16_evolucion_superficie')

### 7.16 Dispersión del margen por temporada

In [ ]:
orden = ['REGULAR','VALENTIN','MADRES','DIFUNTOS','NAVIDAD']
datos = [d.loc[d.season_id == o, 'margin_m2'].values for o in orden]
fig, ax = plt.subplots(figsize=(7.5, 4.4))
bp = ax.boxplot(datos, tick_labels=orden, patch_artist=True, showfliers=False,
                medianprops=dict(color='black', lw=1.4))
for p in bp['boxes']:
    p.set_facecolor(MO); p.set_alpha(.65)
ax.axhline(0, color='k', lw=.9)
ax.set_ylabel('Margen (USD por m² al mes)')
ax.set_title('Dispersión del margen según la temporada comercial')
ax.tick_params(axis='x', rotation=15)
guardar(fig, 'fig17_boxplot_temporada')

### 7.17 Destino de la producción

In [ ]:
comp = pd.Series({
    'Vendida': d.sales.sum(),
    'Nacional (descartada)': d.national.sum(),
    'Exportable sin comprador': d.gross_prod.sum() - d.sales.sum() - d.national.sum()})
comp = comp / comp.sum() * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(comp.index, comp.values, color=[VE, RO, NA])
for i, val in enumerate(comp.values):
    ax.text(val + .6, i, f'{val:.1f}%', va='center')
ax.set_xlabel('Porcentaje de la producción bruta'); ax.set_xlim(0, 100)
ax.set_title('Destino de la producción, 2020-2026')
guardar(fig, 'fig18_destino_produccion')

## 8. Diseño del modelo estadístico

El proyecto adopta una arquitectura secuencial de dos etapas.

**Etapa predictiva.** Un modelo Random Forest estima el componente anticipable del margen mensual por metro cuadrado a partir de rezagos, medias móviles, variables de escala y calendario comercial. Su función no es pronosticar, sino descomponer:

$$\varepsilon_{v,t} = m_{v,t} - \hat{m}_{v,t}$$

**Etapa prescriptiva.** Programación cuadrática de media-varianza sobre los residuos, con la matriz de covarianzas estimada por encogimiento de Ledoit-Wolf:

$$\hat{\Sigma} = \delta F + (1-\delta) S$$

$$\min_w \; w'\hat{\Sigma}w \quad \text{s.a.} \quad \mu'w \geq \theta, \;\; \textstyle\sum_i w_i = 1, \;\; 0 \le w_i \le u_i, \;\; \tfrac{1}{2}\textstyle\sum_i |w_i - w^0_i| \le \tau$$

donde $u_i$ es el techo por variedad y $\tau$ la rotación máxima anual.

In [ ]:
# Verificacion del supuesto que sostiene la agregacion anual del riesgo:
# la anualizacion x12 exige ausencia de dependencia serial.
serie = d.groupby('id_month').apply(lambda g: g.margin.sum() / g.area.sum())
print(f'Autocorrelación de la serie agregada (rezago 1):  {serie.autocorr(1):+.3f}')
print(f'Autocorrelación de la serie agregada (rezago 12): {serie.autocorr(12):+.3f}')

d.to_pickle(SALIDA + 'panel_analitico.pkl')
print(f'\nPanel analítico guardado: {len(d):,} observaciones, {d.variety.nunique()} variedades')